# CS231 - RESNET-18 ABLATION STUDY: SO SÁNH 3 MÔ HÌNH TRÊN TOÀN BỘ DỮ LIỆU
## Đề tài: Nhận diện 6 Mệnh giá Tiền Polymer Việt Nam

Thực nghiệm đối sánh 3 mô hình độc lập (cùng seed 42, cùng lr 1e-4, cùng ResNet-18 ImageNet Pretrained):
1. **Mô hình 1 (Baseline)**: CrossEntropyLoss thuần túy không trọng số (`weight=None`) + Quyết định Argmax.
2. **Mô hình 2 (Cost-Sensitive 1 chiều)**: Weighted CrossEntropyLoss ($w = [0.36, 0.39, 0.49, 0.69, 1.17, 2.67]$) + Quyết định Bayes 1 chiều ($C_{1D}$).
3. **Mô hình 3 (Cost-Sensitive 2 chiều)**: Weighted CrossEntropyLoss ($w = [1.12, 0.87, 0.64, 0.61, 0.83, 1.79]$) + Quyết định Bayes 2 chiều ($C_{2D}$).

Đánh giá song song trên cả:
- **Tập Test độc lập ($1.553$ ảnh)** theo chuẩn `METRICS_STANDARD.md`.
- **Toàn bộ dữ liệu ($15.611$ ảnh)** để có bức tranh toàn cảnh hiệu năng.


In [ ]:
import os
import sys
import time
import json
import urllib.request
import zipfile
from pathlib import Path
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)


In [ ]:
# ĐỘ ĐO CHUẨN CỦA DỰ ÁN (Embedded Self-Contained)
CLASS_ORDER = (10000, 20000, 50000, 100000, 200000, 500000)
DENOMS = CLASS_ORDER
CRITICAL = (3, 4, 5) # 100k, 200k, 500k

def evaluate_classification(y_true, y_pred):
    true = np.asarray(y_true, dtype=np.int64)
    pred = np.asarray(y_pred, dtype=np.int64)
    matrix = confusion_matrix(true, pred, labels=CLASS_ORDER)
    precision, recall, f1, support = precision_recall_fscore_support(
        true, pred, labels=CLASS_ORDER, average=None, zero_division=0
    )
    predicted_count = matrix.sum(axis=0)

    per_class = []
    for i, value in enumerate(CLASS_ORDER):
        tp = int(matrix[i, i])
        per_class.append({
            "denomination_vnd": int(value), "support": int(support[i]),
            "predicted_count": int(predicted_count[i]),
            "tp": tp, "fp": int(predicted_count[i] - tp),
            "fn": int(matrix[i, :].sum() - tp),
            "precision": float(precision[i]), "recall": float(recall[i]),
            "f1": float(f1[i]),
        })

    error_pairs = []
    for i, true_value in enumerate(CLASS_ORDER):
        for j, predicted_value in enumerate(CLASS_ORDER):
            count = int(matrix[i, j])
            if i != j and count > 0:
                gap = abs(true_value - predicted_value)
                error_pairs.append({
                    "true_vnd": true_value, "predicted_vnd": predicted_value,
                    "count": count, "gap_vnd": gap,
                    "total_gap_vnd": count * gap,
                })

    total_gap = int(np.abs(true - pred).sum())
    return {
        "class_order_vnd": list(CLASS_ORDER),
        "n_samples": int(len(true)),
        "confusion_matrix": matrix.tolist(),
        "accuracy": float(np.trace(matrix) / len(true)),
        "macro_f1": float(np.mean(f1)),
        "mae_vnd": total_gap / len(true),
        "total_gap_vnd": total_gap,
        "per_class": per_class,
        "error_pairs": error_pairs,
    }

def _normalize(C, floor=0.0):
    C = np.array(C, float)
    K = len(C)
    off = ~np.eye(K, dtype=bool)
    C[off] /= C[off].mean()
    if floor > 0:
        C[off] = np.maximum(C[off], floor)
        C[off] /= C[off].mean()
    np.fill_diagonal(C, 0.0)
    return C

def value_cost_matrix(denoms=DENOMS, mode="log", under=1.0, over=1.0, floor=0.0):
    v = np.asarray(denoms, float)
    K = len(v)
    C = np.zeros((K, K))
    for y in range(K):
        for k in range(K):
            if y == k:
                continue
            if mode == "abs":
                d = abs(v[y] - v[k])
            elif mode == "log":
                d = abs(np.log(v[y] / v[k]))
            elif mode == "rank":
                d = abs(y - k)
            C[y, k] = d * (under if k < y else over)
    return _normalize(C, floor)

def increasing_importance_matrix(denoms=DENOMS, alpha=0.5, base_mode="log", symmetric=False, floor=0.2):
    v = np.asarray(denoms, float)
    base = value_cost_matrix(denoms, mode=base_mode, floor=floor)
    if symmetric:
        C = base * (np.maximum.outer(v, v) / v.min()) ** alpha
    else:
        C = base * ((v / v.min()) ** alpha)[:, None]
    return _normalize(C)

def bayes_predict(P, C):
    return np.argmin(np.asarray(P) @ np.asarray(C), axis=1)

def class_weights_from_cost(C, class_counts=None):
    C = np.asarray(C, float)
    K = len(C)
    w = np.array([C[y, np.arange(K) != y].mean() for y in range(K)])
    if class_counts is not None:
        n = np.asarray(class_counts, float)
        w = w / ((w * n).sum() / n.sum())
    else:
        w = w / w.mean()
    return w

def evaluate_cost(y, pred, C, critical=CRITICAL):
    y, pred, C = np.asarray(y), np.asarray(pred), np.asarray(C, float)
    costs = np.asarray(C)[y, pred]
    avg_cost = float(costs.mean())
    tp = (y == pred) & np.isin(y, list(critical))
    crit_total = np.isin(y, list(critical)).sum()
    critical_recall = float(tp.sum() / max(crit_total, 1))
    return {
        "avg_cost": avg_cost,
        "critical_recall": critical_recall
    }

print("Đã nạp xong bộ độ đo Value Metrics và Cost Metrics (100% self-contained)!")


In [ ]:
DATA_DIR = Path("/tmp/DATA")
DATA_DIR.mkdir(exist_ok=True, parents=True)

RELEASE_URL = "https://github.com/Chef221/CS231-Computer-Vision-Currency-Classification/releases/download/m0-dataset-2026-11-01"
ZIPS = [
    "data-train-1-of-2.zip",
    "data-train-2-of-2.zip",
    "data-dev.zip",
    "data-test.zip"
]

for zip_name in ZIPS:
    dest_zip = Path("/tmp") / zip_name
    if not dest_zip.exists():
        print(f"Đang tải {zip_name}...")
        url = f"{RELEASE_URL}/{zip_name}"
        urllib.request.urlretrieve(url, dest_zip)
        print(f"Đã tải {zip_name} ({dest_zip.stat().st_size / 1e6:.1f} MB)")
    
    print(f"Đang giải nén {zip_name} vào /tmp...")
    with zipfile.ZipFile(dest_zip, 'r') as zip_ref:
        zip_ref.extractall("/tmp")

print("\nThống kê dữ liệu đã giải nén vào /tmp/DATA:")
total_images = 0
for split in ["train", "dev", "test"]:
    count = len(list((DATA_DIR / split).glob("*/*.png")))
    total_images += count
    print(f"  • Tập {split:<5}: {count:>5} ảnh")
print(f"  ==> TỔNG CỘNG: {total_images:,} ảnh")


In [ ]:
CLASS_TO_IDX = {denom: idx for idx, denom in enumerate(CLASS_ORDER)}
IDX_TO_CLASS = {idx: denom for idx, denom in enumerate(CLASS_ORDER)}

class BanknoteDataset(Dataset):
    def __init__(self, root_dir="/tmp/DATA", splits=("train",), transform=None):
        self.transform = transform
        self.samples = []
        
        for split in splits:
            split_dir = Path(root_dir) / split
            for denom in CLASS_ORDER:
                denom_folder = split_dir / str(denom)
                if not denom_folder.exists():
                    continue
                class_idx = CLASS_TO_IDX[denom]
                for img_name in sorted(os.listdir(denom_folder)):
                    if img_name.lower().endswith(".png"):
                        rel_path = f"{split}/{denom}/{img_name}"
                        img_path = denom_folder / img_name
                        self.samples.append((str(img_path), rel_path, class_idx, denom))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, rel_path, class_idx, denom = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, class_idx, denom, rel_path

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = BanknoteDataset(root_dir="/tmp/DATA", splits=("train",), transform=train_transform)
dev_dataset   = BanknoteDataset(root_dir="/tmp/DATA", splits=("dev",),   transform=eval_transform)
test_dataset  = BanknoteDataset(root_dir="/tmp/DATA", splits=("test",),  transform=eval_transform)
all_dataset   = BanknoteDataset(root_dir="/tmp/DATA", splits=("train", "dev", "test"), transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
all_loader   = DataLoader(all_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Tổng số ảnh Train : {len(train_dataset):>6} ảnh ({len(train_loader)} batches)")
print(f"Tổng số ảnh Dev   : {len(dev_dataset):>6} ảnh ({len(dev_loader)} batches)")
print(f"Tổng số ảnh Test  : {len(test_dataset):>6} ảnh ({len(test_loader)} batches)")
print(f"Tổng TOÀN BỘ data : {len(all_dataset):>6} ảnh ({len(all_loader)} batches)")


In [ ]:
def create_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 6)
    return model

def train_and_select_best(model_name, criterion, train_loader, dev_loader, epochs=10, save_name="model.pth"):
    print("=" * 70)
    print(f"   HUẤN LUYỆN: {model_name}")
    print("=" * 70)
    
    set_seed(42)
    model = create_model().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_macro_f1 = -1.0
    best_mae_vnd = float("inf")
    best_epoch = -1
    checkpoint_path = f"/kaggle/working/{save_name}"
    
    print(f"{'Epoch':<6} {'Train Loss':<12} {'Dev F1':<10} {'Dev Acc':<10} {'Dev MAE (VND)':<16} {'Saved?'}")
    print("-" * 70)
    
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for images, targets, _, _ in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(targets)
            
        train_loss = running_loss / len(train_dataset)
        scheduler.step()
        
        # Dev Validation
        model.eval()
        dev_true_vnd = []
        dev_pred_vnd = []
        with torch.no_grad():
            for images, targets, denoms, _ in dev_loader:
                images = images.to(device)
                outputs = model(images)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                dev_true_vnd.extend(denoms.numpy())
                dev_pred_vnd.extend([IDX_TO_CLASS[p] for p in preds])
                
        dev_metrics = evaluate_classification(dev_true_vnd, dev_pred_vnd)
        dev_f1 = dev_metrics["macro_f1"]
        dev_acc = dev_metrics["accuracy"]
        dev_mae = dev_metrics["mae_vnd"]
        
        is_better = False
        if dev_f1 > best_macro_f1:
            is_better = True
        elif np.isclose(dev_f1, best_macro_f1) and dev_mae < best_mae_vnd:
            is_better = True
            
        saved_str = ""
        if is_better:
            best_macro_f1 = dev_f1
            best_mae_vnd = dev_mae
            best_epoch = epoch
            torch.save(model.state_dict(), checkpoint_path)
            saved_str = f"LƯU (Ep {epoch})"
            
        print(f"{epoch:<6} {train_loss:<12.4f} {dev_f1:<10.4f} {dev_acc:<10.4f} {dev_mae:<16.1f} {saved_str}")
        
    print(f"=> CHECKPOINT {model_name}: Epoch {best_epoch} | F1: {best_macro_f1:.4f} | MAE: {best_mae_vnd:,.1f} đ")
    return checkpoint_path


In [ ]:
train_counts = [2299, 1916, 1646, 1918, 2525, 2190]

# 1. Trọng số 1 chiều: phạt theo mệnh giá thật v_thật (symmetric=False)
C_1way = increasing_importance_matrix(alpha=0.5, symmetric=False)
w_1way_np = class_weights_from_cost(C_1way, train_counts)
weights_1way = torch.tensor(w_1way_np, dtype=torch.float32).to(device)

# 2. Trọng số 2 chiều: phạt đối xứng theo max(v_thật, v_đoán) (symmetric=True)
C_2way = increasing_importance_matrix(alpha=0.5, symmetric=True)
w_2way_np = class_weights_from_cost(C_2way, train_counts)
weights_2way = torch.tensor(w_2way_np, dtype=torch.float32).to(device)

print("CẤU HÌNH TRỌNG SỐ CHO 3 MÔ HÌNH:")
print("1. Baseline  : CrossEntropyLoss (weight = None)")
print("2. CS 1 chiều:", [round(x, 2) for x in w_1way_np.tolist()])
print("3. CS 2 chiều:", [round(x, 2) for x in w_2way_np.tolist()])
print()

# Train Model 1: Baseline (Không dùng Cost-Sensitive)
crit_baseline = nn.CrossEntropyLoss()
ckpt_baseline = train_and_select_best("ResNet-18 (Baseline thuần túy)", crit_baseline, train_loader, dev_loader, epochs=10, save_name="best_resnet18_baseline.pth")

# Train Model 2: Cost-Sensitive 1 chiều
crit_1way = nn.CrossEntropyLoss(weight=weights_1way)
ckpt_1way = train_and_select_best("ResNet-18 (Cost-Sensitive 1 chiều)", crit_1way, train_loader, dev_loader, epochs=10, save_name="best_resnet18_cs_1way.pth")

# Train Model 3: Cost-Sensitive 2 chiều
crit_2way = nn.CrossEntropyLoss(weight=weights_2way)
ckpt_2way = train_and_select_best("ResNet-18 (Cost-Sensitive 2 chiều)", crit_2way, train_loader, dev_loader, epochs=10, save_name="best_resnet18_cs_2way.pth")

print("\nĐÃ HOÀN TẤT HUẤN LUYỆN CẢ 3 MÔ HÌNH TRÊN GPU TESLA T4!")


In [ ]:
def evaluate_model_on_loader(ckpt_path, loader, name="Model"):
    model = create_model().to(device)
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()
    
    y_true_idx = []
    y_true_vnd = []
    rel_paths = []
    probs = []
    
    with torch.no_grad():
        for images, targets, denoms, paths in loader:
            images = images.to(device)
            outputs = model(images)
            p = torch.softmax(outputs, dim=1).cpu().numpy()
            probs.append(p)
            y_true_idx.extend(targets.numpy().tolist())
            y_true_vnd.extend(denoms.numpy().tolist())
            rel_paths.extend(paths)
            
    P = np.vstack(probs)
    return np.array(y_true_idx), np.array(y_true_vnd), P, rel_paths

print("Đã sẵn sàng hàm suy luận xác suất.")


In [ ]:
print("=" * 80)
print("     PHẦN 1: ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP TEST (N = 1.553 ảnh)")
print("=" * 80)

y_idx_base, y_vnd_base, P_base, paths_test = evaluate_model_on_loader("/kaggle/working/best_resnet18_baseline.pth", test_loader, "Baseline")
_, _, P_1w, _ = evaluate_model_on_loader("/kaggle/working/best_resnet18_cs_1way.pth", test_loader, "CS 1-way")
_, _, P_2w, _ = evaluate_model_on_loader("/kaggle/working/best_resnet18_cs_2way.pth", test_loader, "CS 2-way")

# 1. Baseline: Argmax
pred_base_idx = np.argmax(P_base, axis=1)
pred_base_vnd = [IDX_TO_CLASS[p] for p in pred_base_idx]

# 2. CS 1-way: Bayes 1-way
pred_1w_idx = bayes_predict(P_1w, C_1way)
pred_1w_vnd = [IDX_TO_CLASS[p] for p in pred_1w_idx]

# 3. CS 2-way: Bayes 2-way
pred_2w_idx = bayes_predict(P_2w, C_2way)
pred_2w_vnd = [IDX_TO_CLASS[p] for p in pred_2w_idx]

# Metrics
m_test_base = evaluate_classification(y_vnd_base, pred_base_vnd)
m_test_1w   = evaluate_classification(y_vnd_base, pred_1w_vnd)
m_test_2w   = evaluate_classification(y_vnd_base, pred_2w_vnd)

cost_test_base_1w = evaluate_cost(y_idx_base, pred_base_idx, C_1way, critical=CRITICAL)
cost_test_base_2w = evaluate_cost(y_idx_base, pred_base_idx, C_2way, critical=CRITICAL)

cost_test_1w = evaluate_cost(y_idx_base, pred_1w_idx, C_1way, critical=CRITICAL)
cost_test_2w = evaluate_cost(y_idx_base, pred_2w_idx, C_2way, critical=CRITICAL)

test_table = [
    {
        "Mô hình": "1. Baseline (Không Cost-Sensitive)",
        "Accuracy": f"{m_test_base['accuracy']:.4f} ({m_test_base['accuracy']*100:.2f}%)",
        "Macro-F1": f"{m_test_base['macro_f1']:.4f}",
        "MAE theo VNĐ": f"{m_test_base['mae_vnd']:,.1f} đ",
        "Tổng sai lệch": f"{m_test_base['total_gap_vnd']:,} đ",
        "Avg Cost 1-chiều": f"{cost_test_base_1w['avg_cost']:.4f}",
        "Avg Cost 2-chiều": f"{cost_test_base_2w['avg_cost']:.4f}",
        "Critical Recall (100k-500k)": f"{cost_test_base_1w['critical_recall']:.4f}"
    },
    {
        "Mô hình": "2. Cost-Sensitive 1 chiều",
        "Accuracy": f"{m_test_1w['accuracy']:.4f} ({m_test_1w['accuracy']*100:.2f}%)",
        "Macro-F1": f"{m_test_1w['macro_f1']:.4f}",
        "MAE theo VNĐ": f"{m_test_1w['mae_vnd']:,.1f} đ",
        "Tổng sai lệch": f"{m_test_1w['total_gap_vnd']:,} đ",
        "Avg Cost 1-chiều": f"{cost_test_1w['avg_cost']:.4f}",
        "Avg Cost 2-chiều": f"{cost_test_1w['avg_cost']:.4f}",
        "Critical Recall (100k-500k)": f"{cost_test_1w['critical_recall']:.4f}"
    },
    {
        "Mô hình": "3. Cost-Sensitive 2 chiều",
        "Accuracy": f"{m_test_2w['accuracy']:.4f} ({m_test_2w['accuracy']*100:.2f}%)",
        "Macro-F1": f"{m_test_2w['macro_f1']:.4f}",
        "MAE theo VNĐ": f"{m_test_2w['mae_vnd']:,.1f} đ",
        "Tổng sai lệch": f"{m_test_2w['total_gap_vnd']:,} đ",
        "Avg Cost 1-chiều": f"{cost_test_2w['avg_cost']:.4f}",
        "Avg Cost 2-chiều": f"{cost_test_2w['avg_cost']:.4f}",
        "Critical Recall (100k-500k)": f"{cost_test_2w['critical_recall']:.4f}"
    }
]

df_test = pd.DataFrame(test_table)
print(df_test.to_string(index=False))


In [ ]:
print("\n" + "=" * 80)
print("     PHẦN 2: ĐÁNH GIÁ TRÊN TOÀN BỘ TẬP DỮ LIỆU (N = 15.611 ảnh)")
print("=" * 80)

y_idx_all, y_vnd_all, P_all_base, paths_all = evaluate_model_on_loader("/kaggle/working/best_resnet18_baseline.pth", all_loader, "All Baseline")
_, _, P_all_1w, _ = evaluate_model_on_loader("/kaggle/working/best_resnet18_cs_1way.pth", all_loader, "All CS 1-way")
_, _, P_all_2w, _ = evaluate_model_on_loader("/kaggle/working/best_resnet18_cs_2way.pth", all_loader, "All CS 2-way")

pred_all_base_idx = np.argmax(P_all_base, axis=1)
pred_all_base_vnd = [IDX_TO_CLASS[p] for p in pred_all_base_idx]

pred_all_1w_idx = bayes_predict(P_all_1w, C_1way)
pred_all_1w_vnd = [IDX_TO_CLASS[p] for p in pred_all_1w_idx]

pred_all_2w_idx = bayes_predict(P_all_2w, C_2way)
pred_all_2w_vnd = [IDX_TO_CLASS[p] for p in pred_all_2w_idx]

m_all_base = evaluate_classification(y_vnd_all, pred_all_base_vnd)
m_all_1w   = evaluate_classification(y_vnd_all, pred_all_1w_vnd)
m_all_2w   = evaluate_classification(y_vnd_all, pred_all_2w_vnd)

cost_all_base_1w = evaluate_cost(y_idx_all, pred_all_base_idx, C_1way, critical=CRITICAL)
cost_all_base_2w = evaluate_cost(y_idx_all, pred_all_base_idx, C_2way, critical=CRITICAL)

cost_all_1w = evaluate_cost(y_idx_all, pred_all_1w_idx, C_1way, critical=CRITICAL)
cost_all_2w = evaluate_cost(y_idx_all, pred_all_2w_idx, C_2way, critical=CRITICAL)

all_table = [
    {
        "Mô hình": "1. Baseline (Không Cost-Sensitive)",
        "Accuracy": f"{m_all_base['accuracy']:.4f} ({m_all_base['accuracy']*100:.2f}%)",
        "Macro-F1": f"{m_all_base['macro_f1']:.4f}",
        "MAE theo VNĐ": f"{m_all_base['mae_vnd']:,.1f} đ",
        "Tổng sai lệch": f"{m_all_base['total_gap_vnd']:,} đ",
        "Avg Cost 1-chiều": f"{cost_all_base_1w['avg_cost']:.4f}",
        "Avg Cost 2-chiều": f"{cost_all_base_2w['avg_cost']:.4f}",
        "Critical Recall (100k-500k)": f"{cost_all_base_1w['critical_recall']:.4f}"
    },
    {
        "Mô hình": "2. Cost-Sensitive 1 chiều",
        "Accuracy": f"{m_all_1w['accuracy']:.4f} ({m_all_1w['accuracy']*100:.2f}%)",
        "Macro-F1": f"{m_all_1w['macro_f1']:.4f}",
        "MAE theo VNĐ": f"{m_all_1w['mae_vnd']:,.1f} đ",
        "Tổng sai lệch": f"{m_all_1w['total_gap_vnd']:,} đ",
        "Avg Cost 1-chiều": f"{cost_all_1w['avg_cost']:.4f}",
        "Avg Cost 2-chiều": f"{cost_all_1w['avg_cost']:.4f}",
        "Critical Recall (100k-500k)": f"{cost_all_1w['critical_recall']:.4f}"
    },
    {
        "Mô hình": "3. Cost-Sensitive 2 chiều",
        "Accuracy": f"{m_all_2w['accuracy']:.4f} ({m_all_2w['accuracy']*100:.2f}%)",
        "Macro-F1": f"{m_all_2w['macro_f1']:.4f}",
        "MAE theo VNĐ": f"{m_all_2w['mae_vnd']:,.1f} đ",
        "Tổng sai lệch": f"{m_all_2w['total_gap_vnd']:,} đ",
        "Avg Cost 1-chiều": f"{cost_all_2w['avg_cost']:.4f}",
        "Avg Cost 2-chiều": f"{cost_all_2w['avg_cost']:.4f}",
        "Critical Recall (100k-500k)": f"{cost_all_2w['critical_recall']:.4f}"
    }
]

df_all = pd.DataFrame(all_table)
print(df_all.to_string(index=False))


In [ ]:
with open("/kaggle/working/ablation_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "test_comparison": test_table,
        "m_test_base": m_test_base,
        "m_test_1w": m_test_1w,
        "m_test_2w": m_test_2w
    }, f, indent=2, ensure_ascii=False)

with open("/kaggle/working/ablation_full_dataset_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "full_comparison": all_table,
        "m_all_base": m_all_base,
        "m_all_1w": m_all_1w,
        "m_all_2w": m_all_2w
    }, f, indent=2, ensure_ascii=False)

df_test.to_csv("/kaggle/working/ablation_test_summary.csv", index=False)

df_preds = pd.DataFrame({"relative_path": paths_test, "y_true_vnd": y_vnd_base, "y_pred_vnd": pred_2w_vnd})
df_preds.to_csv("/kaggle/working/resnet_test_predictions.csv", index=False)
df_all.to_csv("/kaggle/working/ablation_full_summary.csv", index=False)

# Vẽ ma trận nhầm lẫn Test của cả 3 mô hình
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_info = [
    ("1. Baseline (Unweighted)", m_test_base["confusion_matrix"]),
    ("2. CS 1-way (Asymmetric)", m_test_1w["confusion_matrix"]),
    ("3. CS 2-way (Symmetric)",  m_test_2w["confusion_matrix"])
]
labels = [f"{v//1000}k" for v in CLASS_ORDER]

for ax, (title, cm_list) in zip(axes, models_info):
    cm = np.array(cm_list)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(title, fontsize=12, fontweight="bold")
    tick_marks = np.arange(len(labels))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(labels)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Dự đoán", fontsize=10)
    ax.set_ylabel("Thực tế", fontsize=10)
    for i in range(len(labels)):
        for j in range(len(labels)):
            color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, format(cm[i, j], 'd'), horizontalalignment="center", color=color, fontsize=10)

plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrices_all.png", dpi=300)
plt.close()

print("\nĐÃ HOÀN TẤT VÀ LƯU TOÀN BỘ ARTIFACTS:")
print("  • best_resnet18_baseline.pth")
print("  • best_resnet18_cs_1way.pth")
print("  • best_resnet18_cs_2way.pth")
print("  • ablation_test_metrics.json")
print("  • ablation_full_dataset_metrics.json")
print("  • ablation_test_summary.csv")
print("  • ablation_full_summary.csv")
print("  • confusion_matrices_all.png")
